# Машинное обучение, ФКН ВШЭ

## Практическое домашнее задание 3.1 Базовая генерация признаков

### Общая информация

Дата выдачи: 23.02.2026

Мягкий дедлайн: 12.03.2026 23:59MSK

Жесткий дедлайн: 16.03.2026 23:59MSK

### О задании

В данном задании вы познакомитесь с базовыми подходами для создания новых признаков в табличном машинном обучении. Вам понадобится подумать над тем, зачем мы делаем те или иные преобразования, научиться принимать решения, дающие наилучшие результаты, и узнать, как реализовывать их при помощи библиотек

### Оценивание и штрафы

Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Максимально допустимая оценка за эту часть — 6 баллов. Детальнее про оценивание — в самом конце ноутбука.

Сдавать задание после указанного срока сдачи нельзя. При выставлении неполного балла за задание в связи с наличием ошибок на усмотрение проверяющего предусмотрена возможность исправить работу на указанных в ответном письме условиях.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов (подробнее о плагиате см. на странице курса). Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо считываемые диаграммы.

Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

### Формат сдачи
Задания сдаются через систему Anytask. Инвайт можно найти на странице курса. Присылать необходимо ноутбук с выполненным заданием. Сам ноутбук называйте в формате **homework-practice-03-base-Username.ipynb**, где Username — ваша фамилия.

### **Сетап**

<img style="float: right; padding-right:15px; padding-bottom:10px" src="https://i.postimg.cc/26KqqSb2/pomoika2.png" height=300px width=200px alt="Pomoika 2">
    
В этом домашнем задании мы будем работать с задачей классификации, но сконцентрируемся на том, что приносит не меньшую пользу, чем сами модели — замешиванию данных.

Целевая метрика уже выбрана за нас: мы будем считать $\text{ROC-AUC}$, но не простой, а коэффициент Джини:

$$ \text{Gini} = 2 \cdot \text{ROC-AUC} - 1$$

Конечная цель данного мероприятия — собрать пайплайн машинного обучения от и до, начиная с предобработки данных, заканчивая оптимизацией. При желании это можно доделать до целого пет-проекта, особенно если добавить сбор данных и деплой модели, но в дз этого не будет :(.

Цель конкретно базового ноутбука — познакомить вас с основными преобразованиями и собрать солидный фундамент для преобразований посложнее. 

In [ ]:
from sklearn.metrics import roc_auc_score

def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1.0

#### **Данные**

У вас на руках (на Kaggle) датасет по широко известной в неузких кругах видеоигре Dota 2, скачанный через OpenDota API и заботливо анонимизированный. Если вы не знакомы с игрой — ничего страшного, все необходимое для заданий в базовой части мы подробно опишем.

Нас интересует исход матча — победа или поражение, исходя из совершенно разных факторов (например, чтобы делать ставки на спорт, осуждаем?). Это информация о сессии, игроках, героях, и т д. **в первые 15 минут** после начала матча.

Краткая сводка об игре:

- Dota 2 — командная игра: 5 игроков за Свет (Radiant) против 5 за Тьму (Dire).
- Каждый игрок управляет уникальным героем со своим набором атрибутов и способностей.
- Цель — снести главную постройку на вражеской базе (в простонародье трон).
- В процессе матча игроки добывают золото и опыт, покупают предметы и убивают противников, чтобы стать сильнее.
- Ничьих не бывает, фиксированного таймера нет — матч длится до падения трона.

#### 📌 **Важнейшее замечание**

Предполагается, что у вас уже сложилось понимание:
- как крутить и вертеть данные, чтобы фиты делались твёрдо, трансформы ложились чётко, шейпы датафреймов стакались и нужные джойны джойнились;
- как рисовать читаемые графики;
- как проверять качество модели;

Пожалуйста, следите за этим очень-очень внимательно, иначе рискуете получить штраф и всеобщее порицание в нашем уютном МО-1 чатике (хотя, у этого есть плюсы).

Если возникнут вопросы по игровой части — **не стесняйтесь** спрашивать, гуглить, обращаться к GPT или, прости Господи, дотерам. Знание области — важнейшая составляющая хорошего фича инжиниринга

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Для лучшего понимания, в голубеньких пунктах будет небольшое <font color="#7298ce">**обоснование**</font> того, зачем вообще делается то или иное преобразование (в колабе придётся включать интуицию, там не работает HTML). Вы можете её скипнуть, если всё понятно и без этого

</div>

<div style="border-left: 5px solid #f68c9d; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

В пунктах с розоватой, как у всеми любимого Pudge, расцветкой мы попросим вас написать <font color="#f68c9d">**Ответ**</font> на <font color="#f68c9d">**Вопрос**</font> или <font color="#f68c9d">**рефлексию**</font>, которая должна направить вашу мысль о том, как варить фичи, в нужном направлении. Уметь аргументировать свою точку зрения важно не менее

Пожалуйста, даже если вы уже прожжённый дед инсайд и дата-сайентист 14 уо, всё равно <font color="#f68c9d">**порефлексируйте**</font>. Количество потерянных нервных клеток и ваш успех на соревновании напрямую зависят от базовой предобработки.
Вы **можете** писать <font color="#f68c9d">**её**</font> ёмко, но только если **знаете** ответ

</div>

<div style="border-left: 5px solid #d18753; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(200, 156, 105, 0.05);">

<font>По ходу домашки вам придётся делать <font color="#d18753">**выборы**</font> и подбирать <font color="#d18753">**гиперпараметры**</font>. Какие-то из них важнее остальных. У ряда выборов последствия чисто номинальные, какие-то видоизменят другие задания, третьи полностью определят, как вы будете делать остальную домашку. 

Принимайте решения мудро. Не привязывайтесь к ним слишком сильно, возможно, в процессе вам захочется пересмотреть ваши жизненные приоритеты. Пробуйте, экспериментируйте, фичи это самое творческое, что есть в машинном обучении

При желании, в конце обоих ноутбуков есть инструменты, которые, при остром желании и избытке свободного времени, могут тупо перебрать все выборы и найти самый оптимальный, но это опционально
</div>

### **Часть 1. Это, так сказать, база. (3.25 балла)** <img align="center" height=28 width=28 src="https://media.tenor.com/5vGX5VO-IxsAAAAi/arthas.gif">

В которой студент учится смотреть на фичи под правильным углом и готовить из сырых данных простые, но аппетитные факторы

#### **Задание 1.1. Датасет** (0.5 балла)

Чтобы начать работу с данными, эти данные сперва нужно [загрузить](https://www.kaggle.com/t/6fd940fbeb1746a78031e5d0277f6105). Пока что нам потребуются лишь:

1. Информация о матчах - `matches_df_train.csv`.
2. Тестовые данные для соревнования - `matches_df_test.csv`.

Посмотрите на все csv-файлы выше, создайте под каждый из них отдельный датафрейм и отметьте (текстом или кодом):
- объемы таблиц: как в Мб, так и `df.shape`
- какие в них есть колонки по своему содержанию

In [ ]:
import pandas as pd

train_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/matches_df_train.csv")
test_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/matches_df_test.csv")

In [ ]:
print("matches_df_train")
print(train_df.shape)
display(train_df.sample(10))
display(train_df.describe(include="all"))
display(train_df.isna().sum())

print("matches_df_test")
print(test_df.shape)
display(test_df.sample(10))
display(test_df.describe(include="all"))
display(test_df.isna().sum())

Отдельно хочется посмотреть распределение целевой переменной, покажите его, пожалуйста

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=train_df, x="radiant_win")
plt.title("Распределение radiant_win")
plt.xlabel("radiant_win")
plt.ylabel("count")
plt.show()

<div style="border-left: 5px solid #c27985; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** подходящая ли в данном случае метрика $\text{Gini}$ в сравнении, например, с $\text{AUC-PR}$?

**Ответ:** да, потому что распределение сбалансированно, а $\text{AUC-PR}$ больше подходит для таргета с сильным дисбалансом
</div>

#### **Задание 1.2. Категории** (0.75 балла)

Чтобы построить реально балдёжную модель, зачастую не получится просто написать фит предикт. О нет, это долгая и утомительная возня. А если нужно ещё и отчётики писать, то хоть <span style="color:grey"><font size="1">~~вешайся (осуждаем)~~ </font></span> увольняйся. Так и здесь. И того, что есть, уже хватит, чтобы продемонстрировать глубокую и тёмную сторону Dota Science.

В целом, данные уже содержат признаки, по которым что-то даже можно построить, в частности — регионы.

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

У Radiant, по сравнению с Dire, исторически есть небольшое преимущество — чуть удобнее карта, порядок выбора героев и всё такое. В разных регионах бывают разные предпочтения по стилям игры и тактикам, и где-то это преимущество реализуют лучше

</div>

Посмотрите, где у нас содержится информация о регионе, на серверах которого был проведён матч, и постройте 2 графика:
1) Распределение регионов (процентное и абсолютное) на тренировочных и тестовых данных
2) Среднее значение таргета на трейне, в зависимости от региона

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.countplot(data=train_df, x="region", ax=axes[0, 0])
axes[0, 0].set_title("Train: регионы (абсолютное)")
axes[0, 0].tick_params(axis="x", labelrotation=40)

train_pct = train_df["region"].value_counts(normalize=True).sort_index() * 100
sns.barplot(x=train_pct.index, y=train_pct.values, ax=axes[0, 1])
axes[0, 1].set_title("Train: регионы (процентное)")
axes[0, 1].set_ylabel("percent")
axes[0, 1].tick_params(axis="x", labelrotation=40)

sns.countplot(data=test_df, x="region", ax=axes[1, 0])
axes[1, 0].set_title("Test: регионы (абсолютное)")
axes[1, 0].tick_params(axis="x", labelrotation=40)

test_pct = test_df["region"].value_counts(normalize=True).sort_index() * 100
sns.barplot(x=test_pct.index, y=test_pct.values, ax=axes[1, 1])
axes[1, 1].set_title("Test: регионы (процентное)")
axes[1, 1].set_ylabel("percent")
axes[1, 1].tick_params(axis="x", labelrotation=40)

plt.tight_layout()
plt.show()

In [ ]:
target_by_region = train_df.groupby("region")["radiant_win"].mean().sort_values(ascending=False)
display(target_by_region)
sns.barplot(target_by_region)
plt.xticks(rotation=40)
plt.ylabel("mean radiant_win")
plt.title("Среднее значение таргета по регионам (train)")
plt.tight_layout()
plt.show()


<div style="border-left: 5px solid #c27985; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** что вы можете сказать о распределении регионов? Объясните, получится ли вообще обучить по нему модель, есть ли там сигнал?

**Ответ:** да сигнал однозначно есть. Распределение неравномерное (EW доминирует по кол-ву матчей). Также доля побед radient различается по mean между регионами (от 0.36 до 0.57)
</div>

Наша первая развилка — <font color="#d18753">**выбор**</font>, какой из энкодеров стащить. Рекомендуется брать что-то из `category_encoders`, они похожи на стандартные из `sklearn`, но их больше и применять их проще.

| <font color="#d18753">**One-Hot Encoder**</font> | <font color="#d18753">**Target Encoder**</font> |
| :--- | :--- |
| Превращает категориальный признак в вектор из 0 и 1.  <br> 1 стоит на месте i‑го индекса, если у объекта есть i‑е значение признака. | Кодирует категориальный признак средним значением таргета.  <br> Среднее считается по всем объектам с i‑м значением признака. |

Можно взять и другой, но морально готовьтесь получить $\text{Gini} = 0$
</div>

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">
Без энкодинга использовать категориальные признаки в линейных моделях, увы, нельзя, выбор без выбора 
</div>

In [ ]:
#!pip install -qU category-encoders

<div style="border-left: 5px solid #c27985; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** какой из энкодеров кажется вам наиболее привлекательным? Почему?

**Ответ:** меня больше привлекает Target, потому что он компактнее one-hot и напрямую использует связь с таргетом, что должно лучше сработать имхо
</div>

Закодируйте колонку `region` выбранным вами способом

In [ ]:
from category_encoders import TargetEncoder

X_train = train_df.drop(columns=["radiant_win"]).copy()
y_train = train_df["radiant_win"].astype(int)

X_test = test_df.copy()

enc = TargetEncoder(cols=["region"])
X_train["region_te"] = enc.fit_transform(X_train[["region"]], y_train)["region"]
X_test["region_te"] = enc.transform(X_test[["region"]])["region"]

In [ ]:
X_test.head()

In [ ]:
X_train.head()

#### **Задание 1.3. Даты** (1.25 балла)

Если нам хочется видеть будущее, именно время диктует, что брать можно, а что никак нельзя

Найдите колонку дат на тренировочных данных и:
1. Постройте график доли побед Radiant в зависимости от даты матча
2. Сравните временные диапазоны на трейне и тесте

In [ ]:
tmp = train_df.copy()
tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")

win_rate_daily = tmp.groupby(tmp["date"].dt.date, as_index=False)["radiant_win"].mean()

sns.lineplot(data=win_rate_daily, x="date", y="radiant_win")
plt.xticks(rotation=40)
plt.ylabel("win rate")
plt.title("Доля побед Radiant по датам")
plt.tight_layout()
plt.show()

<div style="border-left: 5px solid #c27985; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** видите ли вы какой-то паттерн в распределении дат? Можете ли вы объяснить при помощи сети Интернет, что там произошло? \
(Подсказка: соревновательные игры периодически обновляются)

**Ответ:** видно как выглядит нормальная игра (с 1 по 5 месяц например) и опираясь на мой ресерч летом 24 года вышел крупный патч, где сделали баланс карты и некоторые герои стали сильнее на Dire стороне, из-за чего просела доля побед Radiant. Но потом, когда игроки приспособились и когда понерфели сильные улучшения доля побед поднялась до $\sim 0.51$

</div>

Теперь давайте что-нибудь повыделяем.

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Даты сами по себе это всегда очень простые фичи, функционал есть в любой библиотеке. Связь с таргетом может быть, но не обязана. Именно даты проверить легко и быстро, зависимости бывают неожиданными. Впрочем, это не единственная причина, не переключайтесь

</div>

С таймстемпом можно делать не так много, кроме базовых манипуляций:

1. Вытащите лежащую на поверхности информацию, например, день и день недели. Хватит и этих двух
2. Посмотрите сами на список возможных признаков, будь то [pandas](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.html) или [polars](https://docs.pola.rs/api/python/stable/reference/expressions/temporal.html), и <font color="#d18753">**либо добавьте**</font> 2 признака, которые, как вам кажется, сработают, <font color="#d18753">**либо поясните**</font>, почему это ничего не даст

<div style="border-left: 5px solid #c27985; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** необходимо ли кодировать признаки из дат? Если да, то какие и как? Если нет, то почему? </font>

**Ответ:** не все. Применю циклическое кодирование (sin/cos) дня day_of_week и month, а day и is_weekend оставлю числовыми, тк в этих случаях вряд ли будет наблюдаться проблема с цикличностью и не так обязательно там будет учитывать реальное расстояние между временнымит промежутками

</div>

3. Закодируйте новые признаки, согласно вашему ответу

In [ ]:
import numpy as np

for df in [X_train, X_test]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["is_weekend"] = (df["dayofweek"] > 4).astype(int)  # 5,6 - сб и вс

    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)  # делим на кол-во дней в неделе
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)  # тут 12 месяцев в году
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)


In [ ]:
X_train.sample(5)

In [ ]:
X_test.sample(5)

Заметил, что в тест только 12 месяц, поэтому потом возможно уберу признак month и доп к нему

Кажется, мы что-то забыли... Ах да, надо бы и модель обучить, вот только без валидации это будет как-то не по-моповски, надо озаботиться этим вопросом.

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

На самом деле временная структура нужна не столь, чтобы вытащить какие-то признаки, сколько, чтобы понять распределение и изменение данных во времени же. Даты играют в этом прямую роль

</div>

<div style="border-left: 5px solid #c27985; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** почему здесь будет не лучшим решением брать случайное разбиение на трейн и валидацию?

**Ответ:** случайное разбиение не подойдет потому что у нас есть временная структура, при рандоме в train могут попасть более поздние матчи, чем в validation, что создает утечку и повысит метрику. Нам нужно предсказывать будущее по прошлому (train на более раннем интервале чем validation)

</div>

Тут выбор у вас решили отобрать, делать мы будем OOT валидацию. Впрочем, какие-то опции ещё остались

<table width="800" border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th width="50%">
      <font color="#d18753">OOT (Out-of-Time)</font>
    </th>
    <th width="50%">
      <font color="#d18753">CV OOT (Cross-Validation Out-of-Time)</font>
    </th>
  </tr>
  <tr>
    <td valign="top">
      Валидация с одной отложенной подвыборкой, <br>
      взятой после трешхолда t. Простая, как палка, <br>
      но если валидация получится грустной — <br>
      аномальной, нетипичной, маленькой, — <br>
      то и метрика ваша тоже будет грустной.
    </td>
    <td valign="top">
      Кросс-валидация с k разбиениями по времени <br>
      с итеративным расширением исходного фолда. <br>
      Оценка метрики сместится куда меньше <br>
      по сравнению с плохим сплитом OOT, <br>
      но это долго, если фолдов много.
    </td>
  </tr>
  <tr>
    <td valign="top">
      <code>sklearn.model_selection.train_test_split</code>
    </td>
    <td valign="top">
      <code>sklearn.model_selection.TimeSeriesSplit</code>
    </td>
  </tr>
</table>



Настройте любой из видов валидации (<font color="#d18753">**трешхолд**</font> `t` или <font color="#d18753">**число фолдов**</font> <code>n_folds</code></font> подберите сами). Они оба должны показывать качество адекватно, хотя второй теоретически должен быть более обоснован. CV-OOT не даст вам бонусов, но кто знает, за какие крохи Джини придётся бороться на соревновании?

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

order = X_train.sort_values("date").index
X_train = X_train.loc[order].reset_index(drop=True)
y_train = y_train.loc[order].reset_index(drop=True)

ts = TimeSeriesSplit(n_splits=5)

for tr_ind, val_ind in ts.split(X_train):
    assert X_train.iloc[tr_ind]["date"].max() <= X_train.iloc[val_ind]["date"].min()


И вот теперь мы уже наконец-то будем что-то обучать. Моделей классификации мы знаем как минимум две — SVM и логистическую регрессию, но есть нюанс.

Выборы, выборы..:

<table width="800" border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th width="50%">
      <font color="#d18753">Dedicated model</font>
    </th>
    <th width="50%">
      <font color="#d18753">Gradient Descent</font>
    </th>
  </tr>
  <tr>
    <td valign="top">
      Большой разницы между <code>LogisticRegression</code> и <code>LinearSVC</code> <br>
      из <code>sklearn.linear_model</code> на самом деле нет: <br>
      разделяющие поверхности очень похожи, <br>
      оба хорошо оптимизированы специальными солверами.
    </td>
    <td valign="top">
      Градиентный спуск через <code>sklearn.linear_model.SGDClassifier</code> <br>
      с параметрами <code>loss="log_loss"</code> или <code>loss="hinge"</code> — <br>
      очень соблазнительная альтернатива, но!
    </td>
  </tr>
  <tr>
    <td valign="top">
      <font color="#d18753"><b>+</b></font> Обучать и применять их в разы проще, <br>
      фит–предикт делает брр. <br>
      <font color="#d18753"><b>−</b></font> Они используют сразу всю выборку <br>
      для обучения, а в ходе задания наша выборка <br>
      может вырасти раз так в 1000, немало!
    </td>
    <td valign="top">
      <font color="#d18753"><b>+</b></font> Влезет любой датасет. <br>
      <font color="#d18753"><b>+</b></font> Бóльший контроль над процессом обучения. <br>
      <font color="#d18753"><b>−</b></font> Обучать на больших данных <br>
      (скорее про часть <b>advanced</b>) придётся через <code>partial_fit</code>, неудобно. <br>
      <font color="#d18753"><b>−</b></font> Нужно подбирать больше гиперпараметров.
    </td>
  </tr>
</table>


Выберите <font color="#d18753">**одну из**</font> моделей выше (хотя `LinearRegression` <font color="#d18753">**тоже можно**</font>, если у вас сегодня авантюрное настроение, успех не гарантирован, о рисках узнаете на лекции) **(обращаем внимание, что другие варианты запрещены)**. Обучите по одному экземпляру на группах признаков:

- дат
- регионов
- дат и регионов

Ну и замерьте качество!

<font color="#d18753">**NB**</font>: 

1. Вы сразу можете строить роскошный пайплайн обучения, а не делать по кускам в отдельных блоках, про это есть **advanced** пункт (**6.1**) на 0.5 баллов
2. Если у вас есть GPU, то почему бы его и не [использовать](https://docs.rapids.ai/api/cuml/stable/)? Если гпу у вас нет, у вас теперь точно есть Kaggle, который щедро дарит 30 часов гпу в неделю, пользуйтесь на здоровье, за это есть маленький, но небольшой буст на 0.25 балла (**пункт 6.3**)
3. Наконец, если вы сразу оформите хранилище для результатов запусков ваших моделей, вы снова получите 0.25 балла (**пункт 6.2**)

In [ ]:
from sklearn.linear_model import LogisticRegression


def lr_gini(use_dates, use_region, ts):
    scores = []

    for tr_ind, val_ind in ts.split(X_train):
        X_tr = X_train.iloc[tr_ind].copy()
        X_val = X_train.iloc[val_ind].copy()
        y_tr = y_train.iloc[tr_ind]
        y_val = y_train.iloc[val_ind]

        cols = []
        if use_dates:
            cols += [
                "day",
                "is_weekend",
                "dow_sin",
                "dow_cos",
                "month_sin",
                "month_cos",
            ]
        if use_region:
            enc = TargetEncoder(cols=["region"])
            X_tr["region_te"] = enc.fit_transform(X_tr[["region"]], y_tr)["region"]
            X_val["region_te"] = enc.transform(X_val[["region"]])["region"]
            cols += ["region_te"]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_tr[cols], y_tr)
        p_val = model.predict_proba(X_val[cols])[:, 1]
        scores.append(gini(y_val, p_val))

    return np.mean(scores), np.std(scores)


for type, use_dates, use_region in [
    ("dates", True, False),
    ("regions", False, True),
    ("dates+regions", True, True),
]:
    m, s = lr_gini(use_dates, use_region, ts)
    print(f"{type}: mean_gini={m:.5f}, std={s:.5f}")

<div style="border-left: 5px solid #c27985; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** помогли ли даты? А должны? Хотите ли вы их оставить?

**Ответ:** нет, модель на датах дает маленькое значение метрики gini, при добавлении дат к region качество даже просело, поэтому я считаю, что их лучше удалить

</div>

In [ ]:
X_train = X_train.drop(columns=["date", "day", "dayofweek", "month", "is_weekend", "dow_sin", "dow_cos", "month_sin", "month_cos"])
X_test  = X_test.drop(columns=["date", "day", "dayofweek", "month", "is_weekend", "dow_sin", "dow_cos", "month_sin", "month_cos"])

In [ ]:
X_train.head()

In [ ]:
X_test.head()

#### **Задание 1.4. Числа** (0.75 балла)

Остался неотвеченным лишь один вопрос — а что числовые признаки? С ними всё одновременно и проще, и сложнее.

Найдите, где хранится средний ммр матча — это средний рейтинг игроков, которые в нём участвовали, чем выше, тем лучше.

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

От ммра напрямую зависит то, как игроки пользуются естественным преимуществом Radiant. Чем игроки сильнее, тем чаще это должно происходить, в теории

</div>

Сделайте (на трейне, валидации и тесте) вот что :
1. Постройте график распределения ммров
2. Сравните, насколько распределения похожи между собой визуально

In [ ]:
val_all_ind = np.concatenate([val_ind for _, val_ind in ts.split(X_train)])
val_all_ind = np.unique(val_all_ind)

X_val_all = X_train.iloc[val_all_ind].copy()

train_mmr = X_train[["avg_mmr"]].copy()
train_mmr["split"] = "train"

val_mmr = X_val_all[["avg_mmr"]].copy()
val_mmr["split"] = "validation_cv"

test_mmr = X_test[["avg_mmr"]].copy()
test_mmr["split"] = "test"

mmr = pd.concat([train_mmr, val_mmr, test_mmr], ignore_index=True)

In [ ]:
mmr.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.set_title("Распределение avg_mmr", fontsize=15)

sns.histplot(data=mmr, x="avg_mmr", hue="split", kde=True, bins=30, ax=ax)
plt.show()

<div style="border-left: 5px solid #c27985; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** как бы вы описали это распределение в целом, похоже ли оно на что-то знакомое вам? 

**Ответ:** распределение похоже на лог-нормальное с длинным правым хвостом. Train и validation визуально близки, test тоже похож по форме, но меньше по объёму. Это похоже на правду, потому что большинство игроков имеют средний ммр в районе 2000, и большие значения сложнее получить

</div>

Кто-то где-то говорил, что числовые признаки надо бы стандартизировать, чтобы вышло что-то годное. Этот кто-то прав, но как известно, практика — критерий истины

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Но на практике это зависит от данных. Сходимость итеративных алгоритмов улучшается, но может пострадать качество, особенно, если распределение не нормальное, или есть отличие в $\mu$ и $\sigma$ на тесте
 
Однако даже пара пунктов Джини это довольно хороший буст, вы убедитесь, когда приступите к сореве, за них нужно бороться любой ценой, тем более, что это почти бесплатно

</div>

Так или иначе, у нас и распределение то не нормальное, но, к счастью, это решаемо. Сделайте такие преобразования признака ммров $f_{\text{mmr}}$ и нарисуйте их график:

$$f_{\text{mmr}} \mapsto \log( 1+f_{\text{mmr}} ); \qquad f_{\text{mmr}} \mapsto \sqrt{f_{\text{mmr}}}; \qquad f_{\text{mmr}} \mapsto \frac{1}{1+f_{\text{mmr}}}; \qquad f_{\text{mmr}} \mapsto \exp{\log f_{\text{mmr}}};$$

Затем выберите то, что вам нравится больше всего

In [ ]:
s = X_train["avg_mmr"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(np.log1p(s), bins=40, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("log(1+x)")

sns.histplot(np.sqrt(s), bins=40, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("sqrt(x)")

sns.histplot(1/(1+s), bins=40, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("1/(1+x)")

sns.histplot(np.exp(np.log(s[s > 0])), bins=40, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("exp(log(x))")

plt.tight_layout()
plt.show()

По графикам sqrt(x) выглядит лучше всех

Мы как-то раньше не обращали внимание, но шестое чувство подсказывает, что в ммрах есть пропуски. Выкинуть их не получится, потому что на тесте они тоже есть, поэтому выход один — чем-то заполнять. 

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

А пропуски у нас есть, потому что не все игры рейтинговые, там информацию по ммру собрать не удастся

В категориальных признаках пропуски не так важны для линрега, их можно закодировать специальной категорией. В числовых проигнорировать их не получится

В любом случае, лучше дополнительно добавить признак-флаг `mmr_missing`, который говорит, что пропуск там на самом деле есть. <font color="#d18753">**Можете**</font> замерить его влияние, если есть желание, может это полная дичь и там Джини 0.9?

</div>

<div style="border-left: 5px solid #c27985; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** нормально ли в данном случае заполнить пропуски нулём? А чем тогда, если нет?

**Ответ:** нет, потому что обычно в данных mmr >> 0 и это будет проблемой в дальнейшем (модель может принять пропуск за реальный mmr, хотя это не так). Лучше заполнять медианой и добавлять флаг mmr_missing

</div>

Момент истины. Обучите две новые модели: к оптимальному набору фичей из предыдущего пункта добавьте в одном случае фичу без преобразования, а в другом — после преобразования. Зацените эффект на трейне и на тесте

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Помните, что как бы ни было прекрасно и информативно преобразование в вашей голове, ключевое это перформанс на тесте \
В жизни каждого дата-сайентиста бывает такое, что фича, которая ну должна быть клёвой, <i>математически</i>, на практике оказывается той ещё жижей, и такое, увы, тоже нужно отслеживать

</div>

TargetEncoder нужно обучать отдельно на каждом train-фолде, иначе возникает leakage (кодировка использует таргет из validation и завышает метрику)

In [ ]:
def count_mmr(ts, use_sqrt=False):
    train_gini, val_gini = [], []

    for tr_ind, val_ind in ts.split(X_train):
        X_tr = X_train.iloc[tr_ind].copy()
        X_val = X_train.iloc[val_ind].copy()
        y_tr = y_train.iloc[tr_ind]
        y_val = y_train.iloc[val_ind]

        enc = TargetEncoder(cols=["region"])
        X_tr["region_te"] = enc.fit_transform(X_tr[["region"]], y_tr)["region"]
        X_val["region_te"] = enc.transform(X_val[["region"]])["region"]

        mmr_median = X_tr["avg_mmr"].median()
        for df in (X_tr, X_val):
            df["mmr_missing"] = df["avg_mmr"].isna().astype(int)
            df["mmr_filled"] = df["avg_mmr"].fillna(mmr_median)
            df["mmr_feat"] = np.sqrt(df["mmr_filled"]) if use_sqrt else df["mmr_filled"]

        cols = ["region_te", "mmr_missing", "mmr_feat"]
        model = LogisticRegression(max_iter=1000)

        model.fit(X_tr[cols], y_tr)
        p_tr = model.predict_proba(X_tr[cols])[:, 1]
        p_val = model.predict_proba(X_val[cols])[:, 1]

        train_gini.append(gini(y_tr, p_tr))
        val_gini.append(gini(y_val, p_val))

    return np.mean(train_gini), np.std(train_gini), np.mean(val_gini), np.std(val_gini)

raw_tr, raw_tr_std, raw_val, raw_val_std = count_mmr(ts, use_sqrt=False)
sqrt_tr, sqrt_tr_std, sqrt_val, sqrt_val_std = count_mmr(ts, use_sqrt=True)

print(f"RAW  mmr: train_gini={raw_tr:.5f} ± {raw_tr_std:.5f}, val_gini={raw_val:.5f} ± {raw_val_std:.5f}")
print(f"SQRT mmr: train_gini={sqrt_tr:.5f} ± {sqrt_tr_std:.5f}, val_gini={sqrt_val:.5f} ± {sqrt_val_std:.5f}")


Видно, что sqrt немного лучше

### **Часть 2. Векторы** (1.5 балла) <img align="center" height=28 width=28 src="https://static.wikia.nocookie.net/dota2_gamepedia/images/1/17/Emoticon_sick.gif/revision/latest?cb=20180504011850">

В которой студент испытывает вьетнамские флешбеки от дз1, фиксит чужие баги и делает нереально мощную фичу, которую можно полировать до посинения

#### **Задание 2.1. Большая чистка** (0.75 балла)

Пока что мы никак не использовали информацию про героев, а ведь от них напрямую зависит исход матча, их больше 100 штук и все они разные: кто-то сильнее, кто-то слабее, а кто-то красивее :3. Только в данные кто-то нагадил, придётся убирать! Тут придётся ещё разочек освежить `pandas`/`polars`

Датасеты, которые нас интересуют теперь — `player_df.csv` и `Constants.Heroes.csv`. Там есть и данные на трейне, и на тесте, мы их обязательно приджойним, но потом.

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Данные, как и стойло, необходимо чистить. Ранее нам везло и в целом каких-то извращений мы не наблюдали, да и тут сходу не увидим. Т.н. *"выбросы"* это, во-первых, тема отдельного холивара, а во-вторых история про доменное несоответствие, матстат такое не найдёт, но на то у нас есть мозг, верно?

Под доменом имеется в виду контекст, в котором создаются ваши данные, и процессы которого ваши данные описывают (в нашем случае — то, как устроена игра и баланс в ней). Тогда выброс — это то, что в контекст не вписывается, даже если ошибки там нет. Про это весь пункт

</div>

Первое, что нужно отсмотреть - главные ключи. Начнём с игроков. Повертите `account_id`, вас должны смутить как минимум два айдишника.

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Не все игроки делают свой профиль публичным, их айди в таком случае будет анонимизирован

Небольшая часть данных собрана некорректно, айди таких игроков тоже помечен особенным значением

</div>

In [ ]:
player_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/player_df.csv")
heroes_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/Constants.Heroes.csv")

display(player_df.head())
display(heroes_df.head())
print(player_df.shape, heroes_df.shape)

In [ ]:
player_df["account_id"].value_counts().head(5)

Есть два специальных id: 4294967295 (2^32 - 1) и -1

<div style="border-left: 5px solid #c27985; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** можем ли мы себе позволить выкинуть матчи с какими-либо подозрительными айди без большого ущерба данным?

**Ответ:** нет, только с -1, потому что id (4294967295) слишком много, и удаление даст большую потерю данных. Можно добавить флаг что у нас специальное id (4294967295), а -1 удалить

</div>

In [ ]:
player_df = player_df[player_df["account_id"] != -1].copy()

Следующий логический шаг — одинаковых героев быть в одном матче не должно.  

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

В кастомных, т.е. не основных режимах игры, это может быть не так, но нас они не интересуют

</div>

Найдите, есть ли игры, в которых это по какой-то причине не так. Если таких матчей не слишком много, избавьтесь от них

In [ ]:
group_by_match_id = player_df.groupby("match_id")["hero_id"]

bad_match_ids = group_by_match_id.size().loc[lambda x: x != group_by_match_id.nunique()].index
print("bad matches:", len(bad_match_ids))

Удалим этих игроков, тк их мало

In [ ]:
player_df = player_df[~player_df["match_id"].isin(bad_match_ids)].copy()

Вы могли заметить героя-импостера под индексом 0. Если вы посмотрите в `Constants.Heroes.csv`, то его там не найдёте, потому что это тоже ошибка.

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Скорее всего это те, кто не успел выбрать героя, игра могла просто не начаться

Вряд ли у нас получится заполнить пропуски чем-то адекватным — они зависят от игрока и героя, переменных много. Попробовать можно, но точно не в базовой части

</div>

Финальным аккордом:

- посмотрите, что за игроки играли на герое с `hero_id=0`, и выкиньте такие матчи вместе со всеми пропусками
- найдите игроков, которые в одном матче играли одновременно и за команду сил Тьмы (слоты `{128, ..., 132}`), и за команду сил Света (слоты `{0, ..., 4}`), не являясь подозрительными айди из пункта выше, уберите их
- отфильтруйте `player_df`, оставьте только те матчи, которые есть в наших выборках
- проверьте, что в каждом матче ровно 5 игроков за Свет и ровно 5 за Тьму

Можно в любом порядке, но эти вещи нужно проверить, они поломают следующий пункт

In [ ]:
df = player_df.copy()
print("start: ", df.shape)

# только матчи из train/test
all_matches = pd.Index(train_df["match_id"]).union(test_df["match_id"])
df = df[df["match_id"].isin(all_matches)].copy()
print("1) ", df.shape)

# убираем пропуски
df = df.dropna(subset=["match_id", "account_id", "hero_id", "player_slot"]).copy()

# матчи с hero_id == 0 убираем
hero0_matches = df.loc[df["hero_id"].eq(0), "match_id"].unique()
df = df[~df["match_id"].isin(hero0_matches)].copy()
print("2) ", df.shape)

# типизация + игра только за одну сторону
df["side"] = np.where(
    df["player_slot"].between(0, 4), "radiant",
    np.where(df["player_slot"].between(128, 132), "dire", pd.NA)
)
df = df[df["side"].notna()].copy()

pairs_sides = (
    df[df["account_id"] != 4294967295]
    .groupby(["match_id", "account_id"])["side"]
    .nunique()
)
bad_pairs = pairs_sides[pairs_sides > 1].index

pair_index = pd.MultiIndex.from_frame(df[["match_id", "account_id"]])
df = df[~pair_index.isin(bad_pairs)].copy()
print("3) ", df.shape)

# оставляем только 5v5
cnt = df.groupby(["match_id", "side"]).size().unstack(fill_value=0)
good_matches = cnt.index[(cnt.get("radiant", 0) == 5) & (cnt.get("dire", 0) == 5)]
df = df[df["match_id"].isin(good_matches)].copy()
print("4) ", df.shape)

final_cnt = df.groupby(["match_id", "side"]).size().unstack(fill_value=0)
assert (final_cnt["radiant"] == 5).all()
assert (final_cnt["dire"] == 5).all()

player_df_clean = df.drop(columns=["side"]).copy()
print("final:", player_df_clean.shape)


Если вас всё же одолевает паранойя, то ~~я вас понимаю~~ будьте уверены, что если проблемы в `player_df` и остались, на модель они повлияют минимально. Ну а идеала не бывает нигде

#### **Задание 2.2. Энкодер героев** (0.75 балла)

А зачем мы вообще этим занимаемся? Вопрос хороший. План был в том, чтобы закодировать комбинации героев, которые участвуют в матче. Это чуть более сложный признак, чем обычный трансформ, тут придется поколдовать.

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Идея в том, что какие-то комбинации героев эффективнее других. Если понять, что в матче за персонажи, можно сразу же прикинуть шансы на победу одной из сторон

</div>

Ваша задача - закодировать каждый матч вектором вида:

| match_id | hero_1 | hero_2 | hero_3 | ... | hero_n |
| --- | --- | --- | --- | --- | --- |
| 228 | 1 | 0 | -1 | ... | 0 |

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Потом <font color="#d18753">**можно**</font> бахнуть One-Hot и разбить героев на две команды, если не боитесь переобучения. Вещь необязательная

</div>

Потребуются таблицы `player_df.csv` и `matches_df_*.csv`. Для удобства может пригодиться `Constants.Heroes.csv` (индексы оттуда и в `player_df` верные, по ним можно джойнить, но они идут не по порядку, не смотрите на пандасовский айди).

Хочется видеть либо функцию, либо в идеале класс, который вертит фичами вот так:

Каждый элемент в векторе матча `(hero_1, ..., hero_n)` принимает значение 1, если герой был в команде сил Света (слоты `{0, ..., 4}`), и -1, если в команде сил Тьмы (слоты `{128, ..., 132}`). 

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Класс нужен, поскольку при неаккуратной реализации есть риск не влезть в память при трансформе целого датасета. Главное, чтобы была функция, а там можно применять её на батч. 

Либо же <font color="#d18753">**можете**</font> преисполниться спарс матрицами, которые экономно (не) хранят миллионы нулей. Вещь несложная и полезная

</div>

Реализуйте то, что написано

In [ ]:
from scipy.sparse import csr_array


class HeroesEncoder:
    def __init__(
        self, match_col="match_id", hero_col="hero_id", slot_col="player_slot"
    ):
        self.match_col = match_col
        self.hero_col = hero_col
        self.slot_col = slot_col
        self.hero_to_col_ = None
        self.feature_names_ = None

    def fit(self, X, y=None):
        heroes = (
            pd.Series(X[self.hero_col].dropna().astype(int).unique())
            .sort_values()
            .tolist()
        )
        self.hero_to_col_ = {h: i for i, h in enumerate(heroes)}
        self.feature_names_ = [f"hero_{h}" for h in heroes]
        return self

    def transform(self, X, y=None):
        if self.hero_to_col_ is None:
            raise ValueError("Encoder is not fitted.")

        if y is None:
            match_ids = pd.Index(X[self.match_col].drop_duplicates())
        else:
            match_ids = pd.Index(y)

        match_to_row = {m: i for i, m in enumerate(match_ids)}

        work = X[[self.match_col, self.hero_col, self.slot_col]].copy()
        slots = work[self.slot_col].to_numpy()

        side = np.zeros(len(work), dtype=np.int8)
        side[(slots >= 0) & (slots <= 4)] = 1
        side[(slots >= 128) & (slots <= 132)] = -1

        rows = work[self.match_col].map(match_to_row).to_numpy()
        cols = work[self.hero_col].map(self.hero_to_col_).to_numpy()

        valid = (side != 0) & (~pd.isna(rows)) & (~pd.isna(cols))

        rows = rows[valid].astype(np.int32)
        cols = cols[valid].astype(np.int32)
        data = side[valid].astype(np.int8)

        mat = csr_array(
            (data, (rows, cols)),
            shape=(len(match_ids), len(self.hero_to_col_)),
            dtype=np.int8,
        )
        mat.sum_duplicates()
        mat.data = np.sign(mat.data).astype(np.int8)
        return mat

    def get_feature_names_out(self):
        if self.feature_names_ is None:
            raise ValueError("Encoder is not fitted.")
        return np.array(self.feature_names_, dtype=object)


Осталось лишь самое сладкое — проверить фичу в деле. Обучите две модели: одну со всеми фичами, что мы накрутили, и одну только с фичами героев, покажите качество

In [ ]:
from scipy.sparse import csr_matrix, hstack

hero_encoder = HeroesEncoder().fit(player_df_clean)
player_train_all = player_df_clean[player_df_clean["match_id"].isin(X_train["match_id"])]
X_hero_all = hero_encoder.transform(player_train_all, y=X_train["match_id"])

def evaluate_hero_models(ts):
    all_tr_scores, all_val_scores = [], []
    hero_tr_scores, hero_val_scores = [], []

    for tr_ind, val_ind in ts.split(X_train):
        X_tr = X_train.iloc[tr_ind].copy()
        X_val = X_train.iloc[val_ind].copy()
        y_tr = y_train.iloc[tr_ind]
        y_val = y_train.iloc[val_ind]

        te = TargetEncoder(cols=["region"])
        X_tr["region_te"] = te.fit_transform(X_tr[["region"]], y_tr)["region"]
        X_val["region_te"] = te.transform(X_val[["region"]])["region"]

        mmr_med = X_tr["avg_mmr"].median()
        for d in (X_tr, X_val):
            d["mmr_missing"] = d["avg_mmr"].isna().astype(int)
            d["mmr_filled"] = d["avg_mmr"].fillna(mmr_med)
            d["mmr_feat"] = np.sqrt(d["mmr_filled"])

        dense_tr = csr_matrix(X_tr[["region_te", "mmr_missing", "mmr_feat"]].to_numpy())
        dense_val = csr_matrix(X_val[["region_te", "mmr_missing", "mmr_feat"]].to_numpy())

        X_hero_tr = X_hero_all[tr_ind]
        X_hero_val = X_hero_all[val_ind]

        X_all_tr = hstack([dense_tr, X_hero_tr], format="csr")
        X_all_val = hstack([dense_val, X_hero_val], format="csr")

        model_all = LogisticRegression(max_iter=2000)
        model_hero = LogisticRegression(max_iter=2000)

        model_all.fit(X_all_tr, y_tr)
        model_hero.fit(X_hero_tr, y_tr)

        p_all_tr = model_all.predict_proba(X_all_tr)[:, 1]
        p_all_val = model_all.predict_proba(X_all_val)[:, 1]
        p_hero_tr = model_hero.predict_proba(X_hero_tr)[:, 1]
        p_hero_val = model_hero.predict_proba(X_hero_val)[:, 1]

        all_tr_scores.append(gini(y_tr, p_all_tr))
        all_val_scores.append(gini(y_val, p_all_val))
        hero_tr_scores.append(gini(y_tr, p_hero_tr))
        hero_val_scores.append(gini(y_val, p_hero_val))

    print(f"ALL (region+mmr+heroes): train={np.mean(all_tr_scores):.5f} ± {np.std(all_tr_scores):.5f}, "
          f"val={np.mean(all_val_scores):.5f} ± {np.std(all_val_scores):.5f}")
    print(f"HEROES: train={np.mean(hero_tr_scores):.5f} ± {np.std(hero_tr_scores):.5f}, "
          f"val={np.mean(hero_val_scores):.5f} ± {np.std(hero_val_scores):.5f}")

evaluate_hero_models(ts)


Итого у вас должно получиться что-то на уровне $\text{Gini} = 0.25$ на тесте, а может даже выше

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Вы могли заметить, что прирост получится не таким значительным, как можно было ожидать, хотя признак сильный. Чем больше добавлять хороших признаков, тем лучше и лучше они объясняет данные.

Даже сложный и хороший по отдельности признак в комбинации с другими будет давать всё меньшее качество. Отчасти это объясняется мультиколлинеарностью, отчасти природой данных, некоторые из них просто слишком сложны

</div>

### **Часть 3. Оптимизация для уже смешариков (1.25 балла)** <img height=28 width=28 align="center" src="https://cdn.7tv.app/emote/01H8RPMSBR000133946WK71YXM/1x.avif">

В которой студент изучает, как оптимизировать модель **по-взрослому**

#### **Задание 3.1. Optuna для самых маленьких** (0.75 балла)

Не стоит забывать, что у любой модели есть <font color="#d18753">**гиперпараметры**</font>. Конечно, львиная доля качества будет идти от фичей, но списывать их со счетов не стоит. В конце концов, бывает, что с безнадёжным на первый взгляд набором признаков, оптимизированная модель покажет лучшее качество, чем базовая модель на топовых фичах, вот и посмотрим.
<a id="section"></a>

Для эффективного подбора гиперпараметров существует несколько решений, основанных на байесовской оптимизации. Одно из наиболее удобных — [optuna](https://optuna.org/), которая делает перебор гиперпараметров таким же лёгким и увлекательным занятием, как составление домашек по МО.

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Идея — смоделировать, какой набор гиперпараметров даст лучшее качество там, где перспективнее, на основе истории их подбора (подробнее на МО2)

</div>

Напишите функцию оптимизации для вашего классификатора. Можете воспользоваться шаблоном ниже, но он довольно куцый, курите документацию. Раз уж инструмент новый, начнём с чего-то простого. Подберите вот такие гиперпараметры (посмотрите на них, прежде, чем тюнить, у них разные диапазоны и разный же смысл (логарифмическая шкала — наш лучший друг, возможно даже лучше настоящих)):

1) Численный — `alpha` у `SGDClassifier` или параметр регуляризации `C` у всех остальных.
2) Категориальный — `solver` у `LogisticRegression`, `loss` у всех остальных
3) Число итераций — `max_iter`. Это не совсем гиперпараметр, но поверьте, обучать модель 100 лет вы не хотите, к тому же это вид неявной регуляризации

<div style="border-left: 5px solid #7298ce; padding: 10px 20px 1px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Раз уж все алгоритмы у нас итерационные, от того, как и по какой функции оптимизироваться, довольно важно

Способов применять оптуну вновь два. \
<font color="#d18753">**Первое**</font> — тестить после каждой фичи, это точнее, но можем переобучиться. <font color="#d18753"> \
**Второе**</font> — подобрать параметры один раз, это проще, но зато быстро. У нас модель простая и перебирать там можно мало чего, пока что, во второй части параметров станет больше. \
Выбор, как всегда, в ваших руках.

</div>

In [ ]:
import optuna


def objective(trial):
    params = {
        "solver": trial.suggest_categorical("solver", ["liblinear", "lbfgs"]),
        "max_iter": trial.suggest_int("max_iter", 600, 2400, step=200),
        "C": trial.suggest_float("C", 1e-4, 30, log=True),
    }

    fold_scores = []

    for fold_id, (tr_ind, val_ind) in enumerate(ts.split(X_train), start=1):
        X_tr = X_train.iloc[tr_ind].copy()
        X_val = X_train.iloc[val_ind].copy()
        y_tr = y_train.iloc[tr_ind]
        y_val = y_train.iloc[val_ind]

        enc = TargetEncoder(cols=["region"])
        X_tr["region_te"] = enc.fit_transform(X_tr[["region"]], y_tr)["region"]
        X_val["region_te"] = enc.transform(X_val[["region"]])["region"]
        mmr_med = X_tr["avg_mmr"].median()
        for part in (X_tr, X_val):
            part["mmr_missing"] = part["avg_mmr"].isna().astype(int)
            part["mmr_filled"] = part["avg_mmr"].fillna(mmr_med)
            part["mmr_feat"] = np.sqrt(part["mmr_filled"])

        X_dense_tr = csr_matrix(
            X_tr[["region_te", "mmr_missing", "mmr_feat"]].to_numpy()
        )
        X_dense_val = csr_matrix(
            X_val[["region_te", "mmr_missing", "mmr_feat"]].to_numpy()
        )

        X_hero_tr = X_hero_all[tr_ind]
        X_hero_val = X_hero_all[val_ind]

        X_all_tr = hstack([X_dense_tr, X_hero_tr], format="csr")
        X_all_val = hstack([X_dense_val, X_hero_val], format="csr")

        model = LogisticRegression(**params, random_state=42)
        model.fit(X_all_tr, y_tr)
        p_val = model.predict_proba(X_all_val)[:, 1]

        score = gini(y_val, p_val)
        fold_scores.append(score)

        trial.report(score, step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
study.optimize(objective, n_trials=25, show_progress_bar=True)

In [ ]:
print("best_gini:", study.best_value)
print("best_params:", study.best_params)

#### **Задание 3.2. Немножко про интерпретацию** (0.25 балла)

В оптуне лежит целая россыпь визуализаций, как же их не пощупать? Чтобы окончательно убедить вас в ценности и важности регуляризации, выведите график важности гиперпараметров. Оценим, что реально важно, а что пшик

In [ ]:
from optuna.visualization import plot_param_importances, plot_optimization_history

fig_imp = plot_param_importances(study)
fig_imp.show()

fig_hist = plot_optimization_history(study)
fig_hist.show()

<div style="border-left: 5px solid #c27985; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** что вы видите и как вы прокомментируете, какие параметры важнее всего?

**Ответ:** видно, что решающим параметром является C (0.87 по важности), то есть коэф регуляризации больше всего влияет на качество, max_iter почти не влияет (0.01), а solver (0.12). Значит в выбранном диапазоне все модели успевали сойтись.

По истории оптимизации видно плато, то есть качество стабилизировалось около gini = 0.3078, и дальнешие trial не дают прироста. Следовательно для такого набора признаков дальнейший подбор гиперпараметров не даст заметного улучшения и нужно добавлять новые признаки.

</div>

#### **Задание 3.3. Заключение** (0.25 балла)

Фуух, много конечно, ну да так уж вышло, что теперь делать, данные это всегда геморрой. Напоследок:

1. Зафиксируйте оптимальный набор гиперпараметров. 
2. Сохраните либо модель, либо pipeline для будущего себя в части advanced. 
3. Сделайте тестовый submission на [Kaggle](https://www.kaggle.com/t/6fd940fbeb1746a78031e5d0277f6105), если ещё не.

In [ ]:
best_params = study.best_params
best_cv_gini = float(study.best_value)

print("best_params:", best_params)
print("best_cv_gini:", best_cv_gini)

hero_encoder = HeroesEncoder().fit(player_df_clean)

player_train_all = player_df_clean[player_df_clean["match_id"].isin(X_train["match_id"])]
player_test_all = player_df_clean[player_df_clean["match_id"].isin(X_test["match_id"])]

X_hero_train = hero_encoder.transform(player_train_all, y=X_train["match_id"])
X_hero_test = hero_encoder.transform(player_test_all, y=X_test["match_id"])

te_final = TargetEncoder(cols=["region"])
region_train = te_final.fit_transform(X_train[["region"]], y_train)["region"]
region_test = te_final.transform(X_test[["region"]])["region"]

mmr_med = X_train["avg_mmr"].median()
mmr_missing_train = X_train["avg_mmr"].isna().astype(int)
mmr_missing_test = X_test["avg_mmr"].isna().astype(int)
mmr_feat_train = np.sqrt(X_train["avg_mmr"].fillna(mmr_med))
mmr_feat_test = np.sqrt(X_test["avg_mmr"].fillna(mmr_med))

X_region_mmr_train = csr_matrix(np.c_[region_train, mmr_missing_train, mmr_feat_train])
X_region_mmr_test = csr_matrix(np.c_[region_test, mmr_missing_test, mmr_feat_test])

X_all_train = hstack([X_region_mmr_train, X_hero_train], format="csr")
X_all_test = hstack([X_region_mmr_test, X_hero_test], format="csr")

model_final = LogisticRegression(**best_params, random_state=42)
model_final.fit(X_all_train, y_train)

test_pred = model_final.predict_proba(X_all_test)[:, 1]
submission = pd.DataFrame({
    "ID": X_test["match_id"].values,
    "Value": test_pred
})
submission.to_csv("submission_all_features_base.csv", index=False)

print("saved")